# 03 · Comparabilidad entre administraciones

**Proyecto SECOP II Barrancabermeja, capa de análisis comparativo**




In [1]:
# 1. Librerías
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200)

In [2]:
# 2. Detectar rutas del proyecto
RUTA_ACTUAL = Path.cwd().resolve()

if RUTA_ACTUAL.name.lower() == "notebooks":
    RUTA_PROYECTO = RUTA_ACTUAL.parent
elif (RUTA_ACTUAL / "datos").exists():
    RUTA_PROYECTO = RUTA_ACTUAL
else:
    RUTA_PROYECTO = RUTA_ACTUAL.parent

RUTA_INTERMEDIOS = RUTA_PROYECTO / "datos" / "intermedios"
RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"
RUTA_TABLAS = RUTA_PROYECTO / "entregables" / "tablas"

for ruta in [RUTA_INTERMEDIOS, RUTA_PROCESADOS, RUTA_TABLAS]:
    ruta.mkdir(parents=True, exist_ok=True)

print("Proyecto:", RUTA_PROYECTO)

Proyecto: D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\PROYECTO SECOP_BARRANCABERMEJA


In [3]:
# 3. Cargar la base clasificada del cuaderno 02

base = pd.read_parquet(RUTA_INTERMEDIOS / "02_base_maestra_clasificada.parquet")
base["fecha_asignacion_periodo"] = pd.to_datetime(
    base["fecha_asignacion_periodo"], errors="coerce"
)

print(f"Contratos: {len(base):,}  |  Columnas: {len(base.columns)}")

Contratos: 37,574  |  Columnas: 130


In [4]:
# 4. Verificar que el cuaderno 02 ya aportó las variables derivadas necesarias

COLUMNAS_REQUERIDAS_02 = [
    "alcalde",
    "mes_gobierno",
    "anio_gobierno",
    "periodo_electoral",
    "periodo_preelectoral",
    "tipo_anio_electoral",
    "ventana_ley_garantias",
    "dias_a_prox_eleccion",
    "anio_periodo",
    "mes_periodo",
    "es_cps_estricto",
    "tipo_cps_claro",
    "valor_contrato_num",
    "valor_mensual_equivalente"
]

faltantes = [c for c in COLUMNAS_REQUERIDAS_02 if c not in base.columns]

if faltantes:
    raise ValueError(
        "La base del cuaderno 02 no trae estas columnas: "
        + ", ".join(faltantes)
        + ". Vuelva a ejecutar 02_limpieza_calidad.ipynb (versión 02_v5 o superior)."
    )

# Confirmar la versión del manifiesto del cuaderno 02
with open(RUTA_INTERMEDIOS / "02_manifiesto_metodologico.json", encoding="utf-8") as f:
    manifiesto_02 = json.load(f)

print("Versión de la base heredada:", manifiesto_02.get("version_base"))
print("Estado de la base:", manifiesto_02.get("estado_base"))
print("Variables del cuaderno 02 verificadas correctamente")

Versión de la base heredada: 02_v5
Estado de la base: APROBADA CON ADVERTENCIAS CONTROLADAS
Variables del cuaderno 02 verificadas correctamente


## Capa 1: Deflactor IPC, de pesos nominales a pesos constantes

SECOP registra **valores nominales** (pesos del año de la firma). La inflación del periodo fue
alta y desigual, así que un mismo honorario real aparece como cifras cada vez mayores con los
años. Sin corregir esto, la administración más reciente parece pagar más solo por efecto de la
inflación.

Se usan como anclas las **variaciones anuales del IPC (diciembre–diciembre) del DANE** y se
interpola un **índice mensual** por crecimiento geométrico entre diciembres. Todos los valores se
llevan a **pesos constantes de mediados de 2025**.

> **Fuente:** DANE, IPC base 2018. La variación de diciembre de 2025 fue **5,10 %**.
> Antes de publicar conviene reemplazar el índice interpolado por la **serie mensual oficial**
> del IPC.


In [5]:
# 5. Parámetros del deflactor (editables). Variación anual DANE, diciembre-diciembre.

INFLACION_DIC = {
    2020: 0.0161,
    2021: 0.0562,
    2022: 0.1312,
    2023: 0.0928,
    2024: 0.0520,
    2025: 0.0510   # dic-2025 confirmado por DANE
}

INFLACION_2026_PROYECTADA = 0.048   # proyección editable para el año en curso
ANIO_BASE = 2025                    # pesos constantes de mediados de este año
MES_BASE = 6

# Índice de diciembre (base dic-2019 = 100)
indice_dic = {2019: 100.0}
for anio in range(2020, 2026):
    indice_dic[anio] = indice_dic[anio - 1] * (1 + INFLACION_DIC[anio])
indice_dic[2026] = indice_dic[2025] * (1 + INFLACION_2026_PROYECTADA)


def indice_ipc_mensual(anio, mes):
    """Índice IPC del mes por interpolación geométrica entre diciembres consecutivos."""
    if pd.isna(anio) or pd.isna(mes):
        return np.nan
    anio, mes = int(anio), int(mes)
    if (anio - 1) not in indice_dic or anio not in indice_dic:
        return np.nan
    inicio, fin = indice_dic[anio - 1], indice_dic[anio]
    return inicio * (fin / inicio) ** (mes / 12.0)


INDICE_BASE = indice_ipc_mensual(ANIO_BASE, MES_BASE)

print("Índice IPC de diciembre por año:")
for anio, valor in indice_dic.items():
    print(f"  {anio}: {valor:6.2f}")
print(f"\nÍndice base (mediados de {ANIO_BASE}): {INDICE_BASE:.2f}")

Índice IPC de diciembre por año:
  2019: 100.00
  2020: 101.61
  2021: 107.32
  2022: 121.40
  2023: 132.67
  2024: 139.57
  2025: 146.68
  2026: 153.72

Índice base (mediados de 2025): 143.08


In [6]:
# 6. Aplicar el deflactor

base["ipc_indice"] = [
    indice_ipc_mensual(anio, mes)
    for anio, mes in zip(base["anio_periodo"], base["mes_periodo"])
]
base["deflactor_ipc"] = INDICE_BASE / base["ipc_indice"]

base["valor_contrato_real"] = base["valor_contrato_num"] * base["deflactor_ipc"]
base["valor_mensual_real"] = base["valor_mensual_equivalente"] * base["deflactor_ipc"]

print("Deflactor medio por año (multiplicador a pesos constantes):")
print(
    base.dropna(subset=["deflactor_ipc"])
    .groupby("anio_periodo")["deflactor_ipc"].mean().round(4)
)

Deflactor medio por año (multiplicador a pesos constantes):
anio_periodo
2020    1.4139
2021    1.3570
2022    1.2605
2023    1.1302
2024    1.0477
2025    0.9963
2026    0.9637
Name: deflactor_ipc, dtype: float64


In [7]:
# 7. Reconstruir los universos de análisis ya enriquecidos

cps = base[base["es_cps_estricto"] == True].copy()
cps_alcaldia = cps[
    (cps["es_alcaldia_analisis"] == True)
    & (cps["alcalde"].isin(["Alfonso Eljach", "Jonathan Vasquez"]))
].copy()

print(f"CPS estricto: {len(cps):,}  |  CPS estricto de la Alcaldía: {len(cps_alcaldia):,}")

CPS estricto: 32,778  |  CPS estricto de la Alcaldía: 26,323


## Capa 2: Efecto del calendario electoral

Las variables electorales vienen del cuaderno 02; aquí solo se leen para documentar el
efecto sobre la serie de contratación.

La serie mensual muestra el mecanismo con nitidez: un **pico de adjudicación en junio de 2023**
justo antes de que empiece la ventana de restricción el 29 de junio, seguido de un
**congelamiento casi total entre julio y octubre**. Ese patrón pertenece al **año 4 de Alfonso**
y no tiene equivalente en el periodo de Jonathan (sin elección hasta 2027): mezclarlos en una
misma ventana distorsiona cualquier comparación de volumen.


In [8]:
# 8. Serie mensual de CPS de la Alcaldía alrededor de la elección de 2023

serie_mensual = (
    cps_alcaldia
    .assign(mes=lambda d: d["fecha_asignacion_periodo"].dt.to_period("M").astype(str))
    .groupby("mes")["contrato_llave"].nunique()
)

serie_2023 = serie_mensual[(serie_mensual.index >= "2023-01") & (serie_mensual.index <= "2023-12")]
print("CPS de la Alcaldía por mes en 2023 (año electoral):")
print(serie_2023)

reporte_garantias = (
    cps_alcaldia.groupby(["alcalde", "ventana_ley_garantias"])["contrato_llave"]
    .nunique().reset_index(name="cps")
)
display(reporte_garantias)

CPS de la Alcaldía por mes en 2023 (año electoral):
mes
2023-01     306
2023-02     509
2023-03     379
2023-04     235
2023-05     133
2023-06    2223
2023-08       1
2023-09       1
2023-10       4
2023-11     387
2023-12     252
Name: contrato_llave, dtype: int64


,alcalde,ventana_ley_garantias,cps
0,Alfonso Eljach,False,12805
1,Alfonso Eljach,True,5
2,Jonathan Vasquez,False,13513


## Capa 3: Ventanas comparables alineadas por mandato

| Ventana | Alfonso | Jonathan | ¿Alineada por gobierno? |
|---|---|---|---|
| 24 meses calendario (heredada del 02) | 2022–2023 (meses 25–48) | 2024–2025 (meses 1–24) | **No**  compara fin de mandato contra inicio |
| Año 3 al mismo corte (heredada del 02) | ene–sep 2022 (meses 25–33) | ene–sep 2026 (meses 25–33) | **Sí** |
| **Meses de gobierno 16–33** (definida aquí) | abr-2021 → sep-2022 | abr-2025 → sep-2026 | **Sí**, y más larga (18 meses) |

La ventana calendario de 24 meses se **conserva** por continuidad, pero queda marcada como
**secundaria**. La comparación principal usa ventanas alineadas por mes de gobierno, que es el
único terreno equivalente: la cobertura confiable de SECOP para Alfonso empieza en su mes 16
(abril de 2021), así que los meses 16 a 33 son la intersección real entre ambos mandatos.


In [9]:
# 9. Definir la ventana alineada por mes de gobierno

# Alfonso tiene cobertura confiable desde su mes 16 (abr-2021).
# Jonathan llega hasta su mes 33 (corte 6-sep-2026).
MES_GOB_INICIO_ALINEADA = 16
MES_GOB_FIN_ALINEADA = 33
N_MESES_ALINEADA = MES_GOB_FIN_ALINEADA - MES_GOB_INICIO_ALINEADA + 1

for tabla in (base, cps_alcaldia):
    tabla["ventana_alineada_gob_16_33"] = tabla["mes_gobierno"].between(
        MES_GOB_INICIO_ALINEADA, MES_GOB_FIN_ALINEADA
    )

print(
    f"Ventana alineada: meses de gobierno {MES_GOB_INICIO_ALINEADA}-{MES_GOB_FIN_ALINEADA} "
    f"({N_MESES_ALINEADA} meses por administración)"
)
print()
print("Meses calendario cubiertos por cada alcalde en la ventana alineada:")
print(
    cps_alcaldia.loc[cps_alcaldia["ventana_alineada_gob_16_33"]]
    .groupby("alcalde")["fecha_asignacion_periodo"]
    .agg(["min", "max"])
)

Ventana alineada: meses de gobierno 16-33 (18 meses por administración)

Meses calendario cubiertos por cada alcalde en la ventana alineada:
                        min        max
alcalde                               
Alfonso Eljach   2021-04-13 2022-09-30
Jonathan Vasquez 2025-04-01 2026-09-04


## Comparación nominal vs real

La tabla siguiente calcula **el mismo indicador** (valor mensual mediano de los CPS) en pesos
nominales y en pesos constantes, en cada ventana.

In [10]:
# 10. Comparación nominal vs real del valor mensual, por ventana

def comparar_ventana(df, filtro, etiqueta):
    sub = df[filtro & df["tipo_cps_claro"]]
    resumen = (
        sub.groupby("alcalde")
        .agg(
            cps=("contrato_llave", "nunique"),
            personas=("proveedor_llave", "nunique"),
            vm_nominal=("valor_mensual_equivalente", "median"),
            vm_real=("valor_mensual_real", "median"),
            duracion_mediana=("duracion_meses_exacta", "median")
        )
        .reset_index()
    )
    resumen.insert(0, "ventana", etiqueta)
    return resumen


comparaciones = pd.concat(
    [
        comparar_ventana(
            cps_alcaldia, cps_alcaldia["periodo_comparable_24m"],
            "24m calendario (secundaria)"
        ),
        comparar_ventana(
            cps_alcaldia, cps_alcaldia["periodo_anio3_mismo_corte"],
            "Año 3 mismo corte (alineada)"
        ),
        comparar_ventana(
            cps_alcaldia, cps_alcaldia["ventana_alineada_gob_16_33"],
            "Meses gobierno 16-33 (alineada)"
        )
    ],
    ignore_index=True
)


def brecha(tabla, columna):
    pivote = tabla.pivot(index="ventana", columns="alcalde", values=columna)
    return ((pivote["Jonathan Vasquez"] / pivote["Alfonso Eljach"] - 1) * 100).round(1)


resumen_brechas = pd.DataFrame({
    "brecha_nominal_%": brecha(comparaciones, "vm_nominal"),
    "brecha_real_%": brecha(comparaciones, "vm_real")
})

print("Valor mensual mediano por ventana (nominal vs real):")
display(comparaciones.round(0))
print("\nBrecha Jonathan vs Alfonso (+ = Jonathan mayor):")
display(resumen_brechas)

Valor mensual mediano por ventana (nominal vs real):


,ventana,alcalde,cps,personas,vm_nominal,vm_real,duracion_mediana
0,24m calendario (secundaria),Alfonso Eljach,8987,4284,2705778.0,3282931.0,4.0
1,24m calendario (secundaria),Jonathan Vasquez,8581,4046,3493115.0,3492935.0,3.0
2,Año 3 mismo corte (alineada),Alfonso Eljach,3288,2638,2698091.0,3481227.0,4.0
3,Año 3 mismo corte (alineada),Jonathan Vasquez,4080,3073,3069580.0,2982502.0,4.0
4,Meses gobierno 16-33 (alineada),Alfonso Eljach,6907,3427,2638133.0,3481227.0,3.0
5,Meses gobierno 16-33 (alineada),Jonathan Vasquez,7629,4212,3044000.0,2982502.0,3.0



Brecha Jonathan vs Alfonso (+ = Jonathan mayor):


,brecha_nominal_%,brecha_real_%
ventana,,
24m calendario (secundaria),29.1,6.4
Año 3 mismo corte (alineada),13.8,-14.3
Meses gobierno 16-33 (alineada),15.4,-14.3


In [11]:
# 11. Volumen, personas y recurrencia en la ventana alineada

alineada = cps_alcaldia[cps_alcaldia["ventana_alineada_gob_16_33"]]

personas = (
    alineada.groupby(["alcalde", "proveedor_llave"])
    .agg(
        contratos=("contrato_llave", "nunique"),
        valor_real=("valor_contrato_real", "sum")
    )
    .reset_index()
)

resumen_alineada = (
    personas.groupby("alcalde")
    .agg(
        personas=("proveedor_llave", "nunique"),
        contratos=("contratos", "sum"),
        contratos_por_persona=("contratos", "mean"),
        pct_2_o_mas=("contratos", lambda x: (x >= 2).mean() * 100)
    )
    .reset_index()
)

resumen_alineada["cps_por_mes"] = (
    resumen_alineada["contratos"] / N_MESES_ALINEADA
).round(1)
resumen_alineada["personas_por_mes"] = (
    resumen_alineada["personas"] / N_MESES_ALINEADA
).round(1)

print(f"Ventana alineada (meses de gobierno {MES_GOB_INICIO_ALINEADA}-{MES_GOB_FIN_ALINEADA}):")
display(resumen_alineada.round(2))

Ventana alineada (meses de gobierno 16-33):


,alcalde,personas,contratos,contratos_por_persona,pct_2_o_mas,cps_por_mes,personas_por_mes
0,Alfonso Eljach,3677,7649,2.08,58.69,424.9,204.3
1,Jonathan Vasquez,4348,7961,1.83,50.92,442.3,241.6


## Guardado de resultados y manifiesto v5

Se guardan la **base enriquecida** con valores reales y la ventana alineada, y las **tablas de
comparación** listas para graficar. 


In [12]:
# 12. Guardar base enriquecida y tablas

base.to_parquet(RUTA_PROCESADOS / "03_base_analitica_enriquecida.parquet", index=False)
cps_alcaldia.to_parquet(RUTA_PROCESADOS / "03_cps_alcaldia_enriquecido.parquet", index=False)

comparaciones.to_csv(
    RUTA_TABLAS / "03_comparacion_valor_mensual_nominal_vs_real.csv",
    index=False, encoding="utf-8-sig"
)
resumen_brechas.to_csv(
    RUTA_TABLAS / "03_brechas_nominal_vs_real.csv", encoding="utf-8-sig"
)
resumen_alineada.to_csv(
    RUTA_TABLAS / "03_resumen_ventana_alineada.csv",
    index=False, encoding="utf-8-sig"
)
serie_2023.to_csv(
    RUTA_TABLAS / "03_serie_cps_alcaldia_2023_electoral.csv", encoding="utf-8-sig"
)

print("Bases y tablas guardadas")

Bases y tablas guardadas


In [13]:
# 13. Manifiesto de la capa de comparabilidad

manifiesto_03 = {
    "version_base": "03_v1",
    "hereda_de": manifiesto_02.get("version_base"),
    "corte_datos": manifiesto_02.get("corte_datos"),
    "alcance": (
        "Capa de analisis comparativo. No limpia ni reclasifica contratos y no recalcula "
        "atributos del contrato: los toma del cuaderno 02."
    ),
    "deflactor_ipc": {
        "fuente": "DANE - IPC base 2018, variacion anual diciembre-diciembre",
        "inflacion_dic": INFLACION_DIC,
        "inflacion_2026_proyectada": INFLACION_2026_PROYECTADA,
        "anio_base_pesos_constantes": ANIO_BASE,
        "metodo": "Indice mensual por interpolacion geometrica entre diciembres",
        "pendiente": (
            "Reemplazar por la serie mensual oficial del IPC antes de publicar "
            "(sustituir la columna ipc_indice)."
        )
    },
    "ventanas": {
        "24m_calendario": "SECUNDARIA. No alineada por mes de gobierno.",
        "anio3_mismo_corte": "Alineada (meses de gobierno 25-33).",
        "gob_16_33": (
            f"Alineada, {N_MESES_ALINEADA} meses (meses de gobierno "
            f"{MES_GOB_INICIO_ALINEADA}-{MES_GOB_FIN_ALINEADA}). Comparacion principal."
        )
    },
    "reglas_de_comparacion": [
        "Los valores monetarios se comparan SOLO en pesos constantes.",
        "Los volumenes se comparan SOLO en ventanas alineadas por mes de gobierno.",
        "La escasez de registros de 2020-2021 es adopcion tardia de SECOP, no menor contratacion.",
        "Los picos y caidas alrededor de octubre de 2023 responden al calendario electoral."
    ]
}

with open(
    RUTA_INTERMEDIOS / "03_manifiesto_comparabilidad.json", "w", encoding="utf-8"
) as archivo:
    json.dump(manifiesto_03, archivo, ensure_ascii=False, indent=2)

print("Manifiesto de comparabilidad guardado")

Manifiesto de comparabilidad guardado
